# Modelo final para predicción de demanda por estación

En este cuaderno entrenamos el modelo ganador y usamos la métrica WAPE/Accuracy como métrica principal de negocio.

Fórmula: 

WAPE = sum(abs(real - predicción)) / sum(real)
Accuracy = 100 × max(0, 1 - WAPE)

La variable objetivo es la demanda por estación y hora: `demand`.
El objetivo final es predecir la demanda de las 12 estaciones con un split temporal train / val / test.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from pulso_transmi import PulsoTransmiClient

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [2]:
client = PulsoTransmiClient()
stations = client.stations()
observations = client.observations_dataframe(page_size=5000)
context = client.context_dataframe(page_size=5000)

df = observations.merge(context, on="observed_at", how="left").merge(stations, on="station_id", how="left")
df["observed_at"] = pd.to_datetime(df["observed_at"]).dt.tz_localize(None)
df = df.sort_values("observed_at").reset_index(drop=True)

# Variables temporales
df["hour"] = df["observed_at"].dt.hour
df["day_of_week"] = df["observed_at"].dt.dayofweek
df["month"] = df["observed_at"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Features lag y rolling para captar la estructura temporal
for lag in [1, 2, 3, 6, 12, 24]:
    df[f"lag_{lag}"] = df.groupby("station_id")["demand"].shift(lag)

for window in [3, 6, 12, 24]:
    df[f"rolling_mean_{window}"] = df.groupby("station_id")["demand"].transform(
        lambda s: s.shift(1).rolling(window, min_periods=1).mean()
    )

for col in [c for c in df.columns if c.startswith("lag_") or c.startswith("rolling_mean_")]:
    df[col] = df[col].fillna(df[col].median())

print("Datos preparados:", df.shape)
print(df[["station_id", "observed_at", "demand", "hour", "day_of_week", "month"]].head())

Datos preparados: (51840, 26)
  station_id         observed_at  demand  hour  day_of_week  month
0      02300 2026-07-26 05:00:00      56     5            6      7
1      10009 2026-07-26 05:00:00     251     5            6      7
2      09122 2026-07-26 05:00:00      59     5            6      7
3      09000 2026-07-26 05:00:00      81     5            6      7
4      07107 2026-07-26 05:00:00      39     5            6      7


In [3]:
target = "demand"
categorical_features = ["station_id", "corridor"]
numeric_features = [
    "rain_mm", "rain_forecast", "temperature_c", "temperature_forecast", "event_intensity",
    "hour", "day_of_week", "month", "is_weekend",
] + [f"lag_{lag}" for lag in [1, 2, 3, 6, 12, 24]] + [f"rolling_mean_{window}" for window in [3, 6, 12, 24]]

features = categorical_features + numeric_features
train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

X_train = train_df[features]
y_train = train_df[target]
X_val = val_df[features]
y_val = val_df[target]
X_test = test_df[features]
y_test = test_df[target]

print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

train=36288, val=7776, test=7776


In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numeric_features,
        ),
    ],
    remainder="drop",
)

def wape_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))
    accuracy = max(0.0, 100.0 * (1.0 - wape))
    return float(wape), float(accuracy)

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    wape, acc = wape_accuracy(y_true, y_pred)
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "r2": r2_score(y_true, y_pred),
        "wape": wape,
        "accuracy": acc,
    }

print("Funciones de evaluación listas.")

Funciones de evaluación listas.


In [5]:
selected_params = {
    "n_estimators": 700,
    "max_depth": 12,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "random_state": 42,
    "n_jobs": -1,
}

final_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(**selected_params)),
    ]
)

print("Modelo final preparado con RandomForest.")
print(selected_params)

Modelo final preparado con RandomForest.
{'n_estimators': 700, 'max_depth': 12, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1}


In [6]:
final_model.fit(X_train, y_train)

val_pred = final_model.predict(X_val)
test_pred = final_model.predict(X_test)

val_metrics = metrics(y_val, val_pred)
test_metrics = metrics(y_test, test_pred)

print("Métricas en validación:")
print({k: round(v, 4) for k, v in val_metrics.items()})
print("\nMétricas en test:")
print({k: round(v, 4) for k, v in test_metrics.items()})

Métricas en validación:
{'mae': 46.183, 'rmse': np.float64(74.8382), 'r2': 0.9431, 'wape': 0.1322, 'accuracy': 86.7773}

Métricas en test:
{'mae': 47.2936, 'rmse': np.float64(75.2029), 'r2': 0.9471, 'wape': 0.1282, 'accuracy': 87.1773}


In [7]:
test_results = test_df[["station_id", "observed_at", target]].copy()
test_results["pred"] = test_pred

station_predictions = (
    test_results.groupby("station_id", as_index=False)
    .agg(
        demanda_real=("demand", "mean"),
        demanda_pred=("pred", "mean"),
    )
)
station_predictions["abs_error"] = np.abs(station_predictions["demanda_real"] - station_predictions["demanda_pred"])
station_predictions["wape_station"] = station_predictions["abs_error"] / station_predictions["demanda_real"].abs()

print("Predicción promedio por estación en test:")
print(station_predictions.sort_values("station_id").round(3).to_string(index=False))

Predicción promedio por estación en test:
station_id  demanda_real  demanda_pred  abs_error  wape_station
     02300       310.599       307.280      3.319         0.011
     03000       269.077       269.320      0.243         0.001
     05000       347.662       346.541      1.121         0.003
     05100       609.140       597.544     11.596         0.019
     06000       522.625       517.172      5.453         0.010
     06111       251.511       254.224      2.713         0.011
     07105       283.049       284.845      1.795         0.006
     07107       298.489       296.599      1.891         0.006
     07111       700.608       689.441     11.167         0.016
     09000       223.611       228.199      4.588         0.021
     09122       259.370       261.069      1.698         0.007
     10009       350.185       348.741      1.444         0.004


In [8]:
# Guardado del modelo
model_dir = Path("../models")
model_dir.mkdir(exist_ok=True)

with open(model_dir / "random_forest_pulso_transmi.json", "w", encoding="utf-8") as f:
    json.dump({
        "selected_model": "RandomForest",
        "selected_params": selected_params,
        "test_metrics": test_metrics,
        "val_metrics": val_metrics,
    }, f, indent=2)

print("Modelo y métricas guardados en ../models/random_forest_pulso_transmi.json")

Modelo y métricas guardados en ../models/random_forest_pulso_transmi.json


## Conclusión

Este cuaderno entrena el modelo seleccionado con la métrica principal WAPE/Accuracy, pero conserva también RMSE, MAE y R² para comparar el desempeño de manera más estándar.

La métrica clave de negocio es:

- WAPE = sum(abs(real - predicción)) / sum(real)
- Accuracy = 100 × max(0, 1 - WAPE)

Esto permite interpretar la calidad del pronóstico en términos de proporción del error respecto a la demanda total.